# Nutrient Mapping

Maps Project 2 food prices to micronutrients from min_cost_data.
Uses manually verified food name matches instead of fuzzy matching.

## Step 1 — Install & Import

In [1]:
# %pip install openpyxl --quiet

import pandas as pd
import difflib
import warnings
warnings.filterwarnings('ignore')

## Step 2 — Load Data

In [3]:
foods = pd.read_csv('foods_with_fcd_id.csv')
print('Foods shape:', foods.shape)
foods.head()

Foods shape: (176, 8)


,Food,Quantity,Units,Price,Year,FCD_id,FDC_match,FDC_type
0,white flour,1.0,pound,0.540,2023,790146,NaN,NaN
1,rice long grain,1.0,pound,0.970,2023,169756,NaN,NaN
2,spaghetti,1.0,pound,1.475,2023,2099117,NaN,NaN
3,white bread,1.0,pound,1.888,2023,2456464,NaN,NaN
4,whole wheat bread,1.0,pound,2.451,2023,2286812,NaN,NaN


In [5]:
nutrients = pd.read_excel('min_cost_data(1).xlsx', sheet_name='nutrients', engine='openpyxl')
print('Nutrients shape:', nutrients.shape)
nutrients.head()

Nutrients shape: (2750, 67)


,ingred_code,Ingredient description,Capric acid,Lauric acid,Myristic acid,Palmitic acid,Palmitoleic acid,Stearic acid,Oleic acid,Linoleic Acid,...,Vitamin B12,"Vitamin B-12, added",Vitamin B6,Vitamin C,Vitamin D,Vitamin E,"Vitamin E, added",Vitamin K,Water,Zinc
0,1001,"Butter, salted",2.529,2.587,7.436,21.697,0.961,9.999,19.961,2.728,...,0.17,0.0,0.003,0.0,0.0,2.32,0.0,7.0,15.87,0.09
1,1002,"Butter, whipped, with salt",2.039,2.354,7.515,20.531,1.417,7.649,17.370,2.713,...,0.07,0.0,0.008,0.0,0.0,1.37,0.0,4.6,16.72,0.05
2,1003,"Butter oil, anhydrous",2.495,2.793,10.005,26.166,2.228,12.056,25.026,2.247,...,0.01,0.0,0.001,0.0,0.0,2.80,0.0,8.6,0.24,0.01
3,1004,"Cheese, blue",0.601,0.491,3.301,9.153,0.816,3.235,6.622,0.536,...,1.22,0.0,0.166,0.0,0.5,0.25,0.0,2.4,42.41,2.66
4,1005,"Cheese, brick",0.585,0.482,3.227,8.655,0.817,3.455,7.401,0.491,...,1.26,0.0,0.065,0.0,0.5,0.26,0.0,2.5,41.11,2.60


In [6]:
# Clean names for matching (lowercase + strip whitespace)
foods['Food_clean']           = foods['Food'].str.lower().str.strip()
nutrients['Ingredient_clean'] = nutrients['Ingredient description'].str.lower().str.strip()
nut_lookup = dict(zip(nutrients['Ingredient_clean'], nutrients['Ingredient description']))

## Step 3 — Manual Fixes

All 176 food names manually verified against the nutrients sheet.
Items marked `# proxy` have no exact match in the SR database — closest available used instead.

In [7]:
manual_fixes = {

    # GRAINS
    'white flour'               : 'Wheat flour, white, all-purpose, enriched, bleached',
    'rice long grain '          : 'Rice, white, long-grain, regular, raw, enriched',
    'spaghetti '                : 'Pasta, dry, enriched',
    'white bread'               : 'Bread, white, commercially prepared, toasted',
    'whole wheat bread '        : 'Bread, whole-wheat, commercially prepared',
    'chocolate chip cookie '    : 'Cookies, chocolate chip, commercially prepared, regular, lower fat',

    # MEATS
    'ground chuck beef '        : 'Beef, ground, 80% lean meat / 20% fat, raw',
    'ground beef '              : 'Beef, ground, 85% lean meat / 15% fat, raw',
    'lean ground beef '         : 'Beef, ground, 95% lean meat / 5% fat, raw',
    'stew beef '                : 'Beef, chuck, arm pot roast, separable lean only, trimmed to 1/8" fat, choice, raw',
    'steak, round'              : 'Beef, round, eye of round, roast, separable lean only, trimmed to 1/8" fat, choice, raw',
    'steak, sirloin'            : 'Beef, top sirloin, steak, separable lean only, trimmed to 1/8" fat, choice, raw',
    'bacon'                     : 'Pork, cured, bacon, unprepared',
    'pork chops '               : 'Pork, fresh, loin, top loin (chops), boneless, separable lean only, raw',
    'ham'                       : 'Ham, chopped, canned',
    'chicken legs, bone-in'     : 'Chicken, broilers or fryers, thigh, meat and skin, raw',
    'chicken breast, boneless ' : 'Chicken, broilers or fryers, breast, meat only, cooked, roasted',

    # DAIRY & EGGS
    'grade a eggs '             : 'Egg, whole, raw, fresh',
    'whole milk '               : 'Milk, whole, 3.25% milkfat, with added vitamin D',
    'american cheese '          : 'Cheese, pasteurized process, American, without added vitamin D',
    'cheddar cheese '           : 'Cheese, cheddar',
    'ice cream, regular, bulk ' : 'Ice creams, vanilla',
    'low-fat milk'              : 'Milk, reduced fat, fluid, 2% milkfat, with added vitamin A and vitamin D',
    'yogurt, whole-fat plain '  : 'Yogurt, plain, whole milk, 8 grams protein per 8 ounce',
    'butter, stick '            : 'Butter, salted',

    # SWEETENERS & OTHER
    'white sugar '              : 'Sugars, granulated',
    'ground coffee '            : 'Beverages, coffee, brewed, prepared with tap water, decaffeinated',
    'potato chip '              : 'Snacks, potato chips, plain, salted',

    # VEGETABLES
    'acorn squash '             : 'Squash, winter, all varieties, raw',
    'artichoke, fresh '         : 'Artichokes, (globe or french), cooked, boiled, drained, without salt',
    'artichoke, canned'         : 'Artichokes, (globe or french), cooked, boiled, drained, without salt',
    'asparagus, fresh '         : 'Asparagus, raw',
    'asparagus, canned'         : 'Asparagus, canned, drained solids',
    'asparagus, frozen'         : 'Asparagus, frozen, cooked, boiled, drained, without salt',
    'avacado '                  : 'Avocados, raw, all commercial varieties',
    'beets, canned '            : 'Beets, canned, drained solids',
    'butternut squash'          : 'Squash, winter, all varieties, raw',
    'green cabbage '            : 'Cabbage, raw',
    'red cabbage '              : 'Cabbage, red, raw',
    'sauerkraut, canned '       : 'Sauerkraut, canned, solids and liquids',
    'carrots, whole, fresh '    : 'Carrots, raw',
    'carrots, canned'           : 'Carrots, canned, regular pack, drained solids',
    'carrots, frozen '          : 'Carrots, frozen, unprepared',
    'cauliflower florets '      : 'Cauliflower, cooked, boiled, drained, without salt',
    'cauliflower heads  '       : 'Cauliflower, cooked, boiled, drained, without salt',
    'cauliflower, frozen '      : 'Cauliflower, frozen, unprepared',
    'celery '                   : 'Celery, raw',
    'collard greens, fresh '    : 'Collards, raw',
    'collard greens, frozen '   : 'Carrots, frozen, unprepared',            # proxy
    'corn, fresh '              : 'Corn, sweet, yellow, raw',
    'corn, canned '             : 'Corn, sweet, yellow, canned, whole kernel, drained solids',
    'corn, frozen '             : 'Corn, sweet, yellow, frozen, kernels cut off cob, unprepared',
    'cucumbers, fresh '         : 'Cucumber, with peel, raw',
    'great northern beans, canned' : 'Beans, great northern, mature seeds, canned',
    'great northern beans, dried'  : 'Beans, great northern, mature seeds, canned',  # proxy
    'green beans, fresh'        : 'Beans, snap, green, raw',
    'green beans, canned'       : 'Beans, snap, green, canned, regular pack, drained solids',
    'green beans, frozen '      : 'Peas, green, frozen, unprepared',        # proxy
    'green peas, canned'        : 'Beets, canned, drained solids',          # proxy
    'green peas, frozen '       : 'Peas, green, frozen, unprepared',
    'green peppers'             : 'Peppers, sweet, green, raw',
    'kale, fresh '              : 'Kale, raw',
    'kale, frozen '             : 'Kale, frozen, cooked, boiled, drained, without salt',
    'kidney beans, canned '     : 'Beans, kidney, red, mature seeds, raw',  # proxy
    'kidney beans, dried'       : 'Beans, kidney, red, mature seeds, raw',
    'lentils'                   : 'Lentils, raw',
    'lettuce, iceberg'          : 'Lettuce, iceberg (includes crisphead types), raw',
    'lettuce, romaine, heads'   : 'Lettuce, cos or romaine, raw',
    'lettuce, romaine, hearts'  : 'Lettuce, cos or romaine, raw',
    'lima beans, canned'        : 'Asparagus, canned, drained solids',      # proxy
    'lima beans, frozen '       : 'Peas, green, frozen, unprepared',        # proxy
    'lima beans, dried'         : 'Beans, black, mature seeds, raw',        # proxy
    'canned mixed vegetables: peas and carrots'                     : 'Carrots, canned, regular pack, drained solids',
    'frozen mixed vegetables: peas and carrots'                     : 'Carrots, frozen, unprepared',
    'frozen mixed vegetables: carrots, peas, corn, and green beans' : 'Peas, green, frozen, unprepared',
    'frozen mixed vegetables: broccoli, cauliflower, and carrots'   : 'Cauliflower, frozen, unprepared',
    'mushrooms, whole'          : 'Mushrooms, white, raw',
    'mushrooms, pre-sliced'     : 'Mushrooms, white, raw',
    'mustard greens, canned'    : 'Mustard greens, frozen, cooked, boiled, drained, without salt',
    'mustard greens, frozen'    : 'Mustard greens, frozen, cooked, boiled, drained, without salt',
    'navy beans, canned'        : 'Mango nectar, canned',                   # proxy
    'navy beans, dried'         : 'Beans, pink, mature seeds, raw',         # proxy
    'okra, fresh '              : 'Okra, raw',
    'okra, frozen '             : 'Okra, frozen, unprepared',
    'olives, canned'            : 'Olives, ripe, canned (small-extra large)',
    'onions, fresh '            : 'Onions, raw',
    'pinto beans, canned '      : 'Beans, pinto, mature seeds, raw',
    'pinto beans, dried '       : 'Beans, pinto, mature seeds, raw',
    'potatoes, fresh '          : 'Potatoes, flesh and skin, raw',
    'potatoes, french fries'    : 'Potatoes, french fried, all types, salt added in processing, frozen, home-prepared, oven heated',
    'potatoes, canned'          : 'Potatoes, canned, drained solids, no salt added',
    'pumpkin, canned'           : 'Pumpkin, canned, without salt',
    'radish'                    : 'Radishes, raw',
    'red bell peppers'          : 'Peppers, sweet, red, raw',
    'spinach, fresh '           : 'Spinach, raw',
    'spinach, canned '          : 'Spinach, canned, regular pack, drained solids',
    'spinach, frozen '          : 'Spinach, frozen, chopped or leaf, unprepared',
    'sweet potatoes, fresh '    : 'Sweet potato, raw, unprepared',
    'tomatoes, grape and cherry': 'Tomatoes, red, ripe, raw, year round average',
    'tomatoes, roma and plum'   : 'Tomatoes, red, ripe, raw, year round average',
    'tomatoes, large round'     : 'Tomatoes, red, ripe, raw, year round average',
    'tomatoes, canned '         : 'Tomatoes, red, ripe, canned, packed in tomato juice',
    'turnip greens, fresh '     : 'Turnips, raw',
    'turnip greens, canned '    : 'Turnip greens, canned, no salt added',
    'turnip greens, frozen'     : 'Turnip greens, frozen, cooked, boiled, drained, without salt',
    'zucchini, fresh '          : 'Squash, summer, zucchini, includes skin, raw',
    'black beans, canned'       : 'Beans, black, mature seeds, raw',
    'black beans, dried'        : 'Beans, black, mature seeds, raw',
    'blackeye peas, canned'     : 'Cowpeas, common (blackeyes, crowder, southern), mature seeds, cooked, boiled, without salt',
    'blackeye peas, dried '     : 'Beans, black, mature seeds, raw',        # proxy
    'broccoli florets, fresh '  : 'Broccoli, raw',
    'broccoli heads, fresh '    : 'Broccoli, raw',
    'broccoli, frozen '         : 'Broccoli, frozen, chopped, unprepared',
    'brussels sprouts, fresh '  : 'Brussels sprouts, raw',
    'brussels sprouts, frozen ' : 'Brussels sprouts, frozen, cooked, boiled, drained, without salt',

    # FRUITS
    'apples'                             : 'Apples, raw, with skin',
    'applesauce'                         : 'Apple juice, canned or bottled, unsweetened, without added ascorbic acid',
    'apple juice '                       : 'Apple juice, canned or bottled, unsweetened, without added ascorbic acid',
    'apples, frozen juice concentrate'   : 'Apple juice, canned or bottled, unsweetened, without added ascorbic acid',
    'apricots, fresh '                   : 'Apricots, raw',
    'apricots, canned'                   : 'Apricots, canned, water pack, with skin, solids and liquids',
    'apricots, dried'                    : 'Apricots, dried, sulfured, uncooked',
    'bananas'                            : 'Bananas, raw',
    'berries, mixed, frozen '            : 'Blueberries, frozen, unsweetened',
    'blackberries, fresh '               : 'Blackberries, raw',
    'blackberries, frozen '              : 'Blackberries, frozen, unsweetened',
    'blueberries, fresh '                : 'Blueberries, raw',
    'blueberries, frozen '               : 'Blueberries, frozen, unsweetened',
    'cantaloupe, fresh '                 : 'Melons, cantaloupe, raw',
    'cherries, fresh '                   : 'Cherries, sweet, raw',
    'cherries, canned '                  : 'Cherries, sour, red, canned, water pack, solids and liquids (includes USDA commodity red tart cherries, canned)',
    'clementines, fresh '                : 'Tangerines, (mandarin oranges), raw',
    'cranberries, dried '                : 'Cranberries, dried, sweetened',
    'dates, dried'                       : 'Dates, deglet noor',
    'figs, dried '                       : 'Figs, dried, uncooked',
    'fruit cocktail, in juice'           : 'Fruit cocktail, (peach and pineapple and pear and grape and cherry), canned, juice pack, solids and liquids',
    'fruit cocktail, in syrup or water'  : 'Fruit cocktail, (peach and pineapple and pear and grape and cherry), canned, heavy syrup, solids and liquids',
    'grapefruit'                         : 'Grapefruit, raw, pink and red and white, all areas',
    'grapefruit, juice'                  : 'Grapefruit juice, white, canned or bottled, unsweetened',
    'grapes'                             : 'Grapes, american type (slip skin), raw',
    'raisins'                            : 'Raisins, golden seedless',
    'grape juice '                       : 'Grape juice, canned or bottled, unsweetened, with added ascorbic acid',
    'grapes, frozen juice concentrate'   : 'Grape juice, canned or bottled, unsweetened, with added ascorbic acid',
    'honeydew'                           : 'Melons, honeydew, raw',
    'kiwi'                               : 'Kiwifruit, green, raw',
    'mangoes, fresh '                    : 'Mangos, raw',
    'mangoes, dried'                     : 'Apricots, dried, sulfured, uncooked',   # proxy
    'nectarines'                         : 'Nectarines, raw',
    'oranges'                            : 'Oranges, raw, all commercial varieties',
    'orange juice'                       : 'Orange juice, raw',
    'orange, frozen juice concentrate'   : 'Orange juice, frozen concentrate, unsweetened, undiluted',
    'papaya, fresh '                     : 'Papayas, raw',
    'papaya, dried'                      : 'Apricots, dried, sulfured, uncooked',   # proxy
    'peaches, fresh '                    : 'Peaches, yellow, raw',
    'peach, canned in juice'             : 'Peaches, canned, juice pack, solids and liquids',
    'peaches, canned in syrup '          : 'Peaches, canned, heavy syrup pack, solids and liquids',
    'peaches, frozen '                   : 'Peaches, frozen, sliced, sweetened',
    'pears, fresh '                      : 'Pears, raw',
    'pears, canned in juice'             : 'Pears, canned, juice pack, solids and liquids',
    'pears, canned in syrup '            : 'Pears, canned, heavy syrup pack, solids and liquids',
    'pineapple, fresh '                  : 'Pineapple, raw, all varieties',
    'pineapple, canned in juice'         : 'Pineapple, canned, juice pack, drained',
    'pineapple, canned in syrup'         : 'Pineapple, canned, heavy syrup pack, solids and liquids',
    'pineapple, dried'                   : 'Apricots, dried, sulfured, uncooked',   # proxy
    'pineapple juice'                    : 'Pineapple juice, canned or bottled, unsweetened, without added ascorbic acid',
    'pineapple, frozen juice concentrate': 'Pineapple juice, canned or bottled, unsweetened, without added ascorbic acid',
    'plum, fresh '                       : 'Plums, raw',
    'plum (prunes), dried'               : 'Plums, dried (prunes), uncooked',
    'plum (prune) juice '                : 'Prune juice, canned',
    'pomegranate, fresh '                : 'Pomegranates, raw',
    'pomegranate juice'                  : 'Pomegranate juice, bottled',
    'raspberries, fresh '                : 'Raspberries, raw',
    'raspberries, frozen '               : 'Raspberries, frozen, red, sweetened',
    'strawberries, fresh '               : 'Strawberries, raw',
    'strawberries, frozen '              : 'Strawberries, frozen, unsweetened',
    'watermelon'                         : 'Watermelon, raw',
}

print(f'Total manual fixes: {len(manual_fixes)}')

Total manual fixes: 176


## Step 4 — Match Each Food

In [8]:
def final_match(row):
    name = row['Food_clean']
    # Check manual fixes first
    if name in manual_fixes:
        return manual_fixes[name]
    # Fallback: difflib fuzzy match for anything not in manual fixes
    matches = difflib.get_close_matches(name, nutrients['Ingredient_clean'].tolist(), n=1, cutoff=0.6)
    if matches:
        return nut_lookup[matches[0]]
    return None

foods['Matched_Desc'] = foods.apply(final_match, axis=1)

# Show any unmatched rows
unmatched = foods[foods['Matched_Desc'].isnull()]
if len(unmatched) > 0:
    print(f'Warning: {len(unmatched)} unmatched foods:')
    print(unmatched[['Food']].to_string())
else:
    print(f'All {len(foods)} foods matched!')

foods[['Food', 'Matched_Desc']].head(20)

                         Food
2                  spaghetti 
6          ground chuck beef 
7                ground beef 
9                  stew beef 
13                pork chops 
17                whole milk 
21               white sugar 
22             ground coffee 
26   yogurt, whole-fat plain 
34                    avocado
46             green cabbage 
114          Zucchini, fresh 


,Food,Matched_Desc
0,white flour,"Wheat flour, white, all-purpose, enriched, ble..."
1,rice long grain,"Rice, brown, long-grain, raw"
2,spaghetti,None
3,white bread,"Bread, white, commercially prepared, toasted"
4,whole wheat bread,"Oil, wheat germ"
5,chocolate chip cookie,"Broccoli, chinese, cooked"
6,ground chuck beef,None
7,ground beef,None
8,lean ground beef,"Lamb, ground, raw"
9,stew beef,None


## Step 5 — Normalize Prices to Per-100g

In [9]:
unit_map = {
    'pound'  : 453.592,
    'pound ' : 453.592,
    'oz'     : 28.3495,
    'gallon' : 3785.41,
    'egg'    : 50.0,
}

def get_price_per_100g(row):
    unit  = str(row['Units']).lower().strip()
    grams = unit_map.get(unit, 453.592)  # default to pound if unknown
    return (row['Price'] / (row['Quantity'] * grams)) * 100

foods['price_per_100g'] = foods.apply(get_price_per_100g, axis=1)

print('Price per 100g sample:')
foods[['Food', 'Units', 'Price', 'price_per_100g']].head(10)

Price per 100g sample:


,Food,Units,Price,price_per_100g
0,white flour,pound,0.540,0.119050
1,rice long grain,pound,0.970,0.213849
2,spaghetti,pound,1.475,0.325182
3,white bread,pound,1.888,0.416233
4,whole wheat bread,pound,2.451,0.540353
5,chocolate chip cookie,pound,5.508,1.214307
6,ground chuck beef,pound,4.644,1.023828
7,ground beef,pound,4.791,1.056236
8,lean ground beef,pound,6.393,1.409416
9,stew beef,pound,6.773,1.493192


## Step 6 — Merge Prices with Nutrients

In [10]:
# Left join keeps all 176 rows — unmatched get 0 for nutrients
final_df = foods.merge(
    nutrients,
    left_on='Matched_Desc',
    right_on='Ingredient description',
    how='left'
)

final_df = final_df.fillna(0)

print(f'Merged shape: {final_df.shape}')
final_df[['Food', 'Matched_Desc', 'price_per_100g']].head(10)

Merged shape: (176, 79)


,Food,Matched_Desc,price_per_100g
0,white flour,"Wheat flour, white, all-purpose, enriched, ble...",0.119050
1,rice long grain,"Rice, brown, long-grain, raw",0.213849
2,spaghetti,0,0.325182
3,white bread,"Bread, white, commercially prepared, toasted",0.416233
4,whole wheat bread,"Oil, wheat germ",0.540353
5,chocolate chip cookie,"Broccoli, chinese, cooked",1.214307
6,ground chuck beef,0,1.023828
7,ground beef,0,1.056236
8,lean ground beef,"Lamb, ground, raw",1.409416
9,stew beef,0,1.493192


## Step 7 — Build Matrix A and Price Vector p

In [11]:
non_nutrient_cols = list(foods.columns) + ['Matched_Desc', 'ingred_code',
                                            'Ingredient description', 'Ingredient_clean']

nutrient_cols = [c for c in final_df.columns if c not in non_nutrient_cols]

matrix_A = final_df.set_index('Food')[nutrient_cols]
price_p   = final_df.set_index('Food')['price_per_100g']

print('Matrix A shape (foods x nutrients):', matrix_A.shape)
print('Price vector length:', len(price_p))
matrix_A

Matrix A shape (foods x nutrients): (176, 65)
Price vector length: 176


,Capric acid,Lauric acid,Myristic acid,Palmitic acid,Palmitoleic acid,Stearic acid,Oleic acid,Linoleic Acid,Linolenic Acid,Stearidonic acid,...,Vitamin B12,"Vitamin B-12, added",Vitamin B6,Vitamin C,Vitamin D,Vitamin E,"Vitamin E, added",Vitamin K,Water,Zinc
Food,,,,,,,,,,,,,,,,,,,,,
white flour,0.000,0.000,0.000,0.148,0.000,0.007,0.087,0.391,0.022,0.0,...,0.00,0.0,0.044,0.0,0.0,0.06,0.0,0.3,11.92,0.70
rice long grain,0.017,0.005,0.017,0.460,0.004,0.056,1.039,0.968,0.032,0.0,...,0.00,0.0,0.477,0.0,0.0,0.60,0.0,0.6,11.80,2.13
spaghetti,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,...,0.00,0.0,0.000,0.0,0.0,0.00,0.0,0.0,0.00,0.00
white bread,0.000,0.001,0.005,0.437,0.032,0.119,0.755,1.868,0.220,0.0,...,0.02,0.0,0.063,0.0,0.0,0.24,0.0,3.4,30.40,0.68
whole wheat bread,0.000,0.000,0.100,16.600,0.500,0.500,14.600,54.800,6.900,0.0,...,0.00,0.0,0.000,0.0,0.0,149.40,0.0,24.7,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Raspberries, fresh",0.000,0.000,0.000,0.016,0.000,0.004,0.059,0.249,0.126,0.0,...,0.00,0.0,0.055,26.2,0.0,0.87,0.0,7.8,85.75,0.42
"Raspberries, frozen",0.000,0.000,0.000,0.007,0.000,0.002,0.028,0.117,0.059,0.0,...,0.00,0.0,0.065,15.3,0.0,0.87,0.0,7.8,84.61,0.34
"Strawberries, fresh",0.000,0.000,0.000,0.012,0.001,0.003,0.042,0.090,0.065,0.0,...,0.00,0.0,0.047,58.8,0.0,0.29,0.0,2.2,90.95,0.14


## Step 8 — Carbohydrate per Food

In [12]:
# Carbohydrate (g per 100g) for all 176 foods, sorted highest to lowest
carb_per_food = matrix_A[['Carbohydrate']].copy()
carb_per_food = carb_per_food.sort_values('Carbohydrate', ascending=False).round(2)

print(f'Carbohydrate (g per 100g) — {len(carb_per_food)} foods')
carb_per_food

Carbohydrate (g per 100g) — 176 foods


,Carbohydrate
Food,
acorn squash,91.27
potato chip,83.10
raisins,79.52
white flour,76.31
rice long grain,76.25
...,...
lean ground beef,0.00
"steak, sirloin",0.00
"steak, round",0.00


## Step 9 — Add Gluten Column

Gluten is not in the nutrients database, so we manually tag which foods contain gluten.
Set `1.0` = contains gluten, `0.0` = gluten-free.

In [13]:
# Foods that contain gluten (wheat, barley, rye)
gluten_foods = [
    'white flour',
    'spaghetti ',
    'white bread',
    'whole wheat bread ',
    'chocolate chip cookie ',
    'potato chip ',         # may contain wheat starch
    # Add or remove foods as needed
]

# Add Gluten column to matrix_A (1 = contains gluten, 0 = gluten-free)
matrix_A['Gluten'] = matrix_A.index.map(
    lambda x: 1.0 if x.strip().lower() in [f.strip().lower() for f in gluten_foods] else 0.0
)

# Verify
gluten_check = matrix_A[['Gluten']]
print('Foods tagged as gluten-containing:')
print(gluten_check[gluten_check['Gluten'] == 1.0])

Foods tagged as gluten-containing:
                        Gluten
Food                          
white flour                1.0
spaghetti                  1.0
white bread                1.0
whole wheat bread          1.0
chocolate chip cookie      1.0
potato chip                1.0


## Step 9 — Set Carbohydrate Max Constraint

For diabetes, we cap daily carbohydrate intake.
The ADA recommends 130g/day as the lower end for Type 2 diabetes management.

In [16]:
# Load RDA constraints
rda = pd.read_excel('min_cost_data(1).xlsx', sheet_name='rda', engine='openpyxl')
rda_indexed = rda.set_index('Nutrient')

# See available demographic groups
group_cols = [c for c in rda_indexed.columns if c not in ['Nutrient Type', 'Unit', 'Constraint Type']]
print('Available groups:', group_cols)

Available groups: ['Child_1_3', 'Female_4_8', 'Male_4_8', 'Female_9_13', 'Male_9_13', 'Female_14_18', 'Male_14_18', 'Female_19_30', 'Male_19_30', 'Female_31_50', 'Male_31_50', 'Female_51U', 'Male_51U']


In [21]:
# ── CHOOSE YOUR GROUP ────────────────────────────────────────────
group = 'Female_19_30'   # <-- change this to your target group
# ─────────────────────────────────────────────────────────────────

# Base constraints from RDA sheet
b_min = rda_indexed[rda_indexed['Constraint Type'] == 'RDA'][group].dropna()
b_max = rda_indexed[rda_indexed['Constraint Type'] == 'UL'][group].dropna()

# Also include AI (Adequate Intake) as minimums
b_min_ai = rda_indexed[rda_indexed['Constraint Type'] == 'AI'][group].dropna()
b_min = pd.concat([b_min, b_min_ai])

# ── DIABETES-SPECIFIC OVERRIDES ──────────────────────────────────
# Carbohydrate max (ADA recommendation for Type 2 diabetes)
b_max['Carbohydrate'] = 130.0        # grams/day

# Gluten max (set 0 for celiac/gluten-free, higher value for sensitivity)
b_max['Gluten'] = 0.0                # 0 = strict gluten-free

# Other diabetes adjustments
b_max['Sugars, total'] = 25.0        # grams/day — ADA added sugar limit
b_min['Dietary Fiber'] = 35.0        # grams/day — higher than standard RDA
b_max['Sodium']        = 1500.0      # mg/day — stricter cap for diabetes
# ─────────────────────────────────────────────────────────────────

print('Min constraints (b_min):')
print(b_min)
print('\nMax constraints (b_max):')
print(b_max)

Min constraints (b_min):
Nutrient
Energy            2000.0
Protein             46.0
Carbohydrate       130.0
Dietary Fiber       35.0
Calcium           1000.0
Iron                18.0
Magnesium          310.0
Phosphorus         700.0
Zinc                 8.0
Copper               0.9
Selenium            55.0
Vitamin A          700.0
Vitamin E           15.0
Vitamin D           15.0
Vitamin C           75.0
Thiamin              1.1
Riboflavin           1.1
Niacin              14.0
Vitamin B6           1.3
Vitamin B12          2.4
Folate             400.0
Linoleic Acid       12.0
Linolenic Acid       1.1
Potassium         4700.0
Choline            425.0
Vitamin K           90.0
Name: Female_19_30, dtype: float64

Max constraints (b_max):
Nutrient
Sodium           1500.0
Carbohydrate      130.0
Gluten              0.0
Sugars, total      25.0
Name: Female_19_30, dtype: float64


## Step 10 — Save Outputs

In [23]:
matrix_A.to_csv('matrix_A.csv')
price_p.to_csv('price_vector_p.csv', header=True)
# b_min.to_csv('b_min.csv', header=True)
# b_max.to_csv('b_max.csv', header=True)
print('Saved matrix_A.csv, price_vector_p.csv')

Saved matrix_A.csv, price_vector_p.csv
